# Experimento 3 — Extensión del Hito 2 para rechazo de fauna y sonidos ambientales

**Proyecto 1 de Innovación Tecnológica — Maestría en Inteligencia Artificial Aplicada (MIAA)**  
**Universidad Icesi · Cali, Colombia**  
**Autores responsables del Hito 2:** Carlos Alberto Salamando · Verónica Serna  
**Equipo completo:** Juan Fernando Gutiérrez · Miguel Ángel Paz · Carlos Alberto Salamando · Verónica Serna  
**Instructor:** Prof. Milton Orlando Sarria Paja  

---

## Objetivo y alcance del Experimento 3

De acuerdo con el plan de trabajo ([`docs/plan_trabajo.md`](../docs/plan_trabajo.md)), el **Hito 2** toma como entrada el subconjunto curado de **16 especies de aves del Valle del Magdalena Medio (`6 759` grabaciones, `71.4` horas)** definido en el Hito 1 ([`notebooks/hito1_eda_seleccion_especies.ipynb`](./hito1_eda_seleccion_especies.ipynb)) y ejecuta el pipeline completo de acondicionamiento físico y lógico de las señales acústicas:

1. **Verificación exhaustiva de archivos, cabeceras binarias y etiquetas:** comprobación de existencia en disco, bandera de fin de flujo `EOS` del contenedor OGG Vorbis, frecuencia de muestreo (`32 000 Hz`), número de canales (`1` canal monofónico), duración mínima ($\ge 1.0\text{ s}$) y pertenencia de etiquetas al conjunto cerrado de 16 especies.
2. **Estandarización de señal a `32 kHz` mono (`float32`) y remoción de componente DC:** decodificación de flujos Vorbis a punto flotante en $[-1, 1]$, conversión a monofónico, remuestreo polifásico racional (`scipy.signal.resample_poly`) en caso de frecuencias distintas y centrado a media cero ($x[n] \leftarrow x[n] - \mu_x$).
3. **Filtrado pasa-banda bioacústico de fase cero (`[150 Hz, 14 000 Hz]`):** diseño de un filtro digital IIR **Butterworth de orden $N = 4$** factorizado en **Secciones Bicuadráticas de Segundo Orden (`SOS`)** y aplicado de forma bidireccional (`scipy.signal.sosfiltfilt`, orden efectivo $2N = 8$, pendiente de $-48\text{ dB/octava}$ y desfase nulo $\Delta\phi(\omega) = 0$) para eliminar ruido de viento, motores y red eléctrica ($<150\text{ Hz}$) así como ruido térmico de alta frecuencia ($>14\text{ kHz}$).
4. **Segmentación en clips de `5.0 segundos` (`160 000` muestras) y control de calidad acústica:**
   - Relleno cíclico con ventana coseno (*tapered cyclic tiling*) para grabaciones cortas ($1.0\text{ s} \le T < 5.0\text{ s}$, `425` grabaciones).
   - Extracción de ventanas de $5.0\text{ s}$ para grabaciones de $T \ge 5.0\text{ s}$ (`36 939` ventanas candidatas evaluadas), evaluación de **9 indicadores acústicos** por ventana (`rms_raw`, `rms_filt`, `dbfs`, `peak_amp`, `crest_factor_db`, `clipping_ratio`, `inband_energy_ratio`, `snr_db` por percentiles $P_{90}/P_{10}$ de subtramas de $100\text{ ms}$, y `activity_score`) y selección guiada por actividad vocal de hasta `K = 4` clips activos por grabación (**`20 748` clips acondicionados, `28.82` horas**).
   - Normalización de amplitud pico a `-1.0 dBFS` ($\approx 0.89125$) y almacenamiento físico en `src/data/processed/clips_5s/<partition>/<primary_label>/<clip_id>.wav`.
5. **Particionamiento estratificado `train / val / test` sin fuga de información (*Zero Data Leakage*):** asignación estricta a nivel de grabación (`recording_id`) y grabador (`author`), garantizando que todos los clips de 5 s derivados de una misma grabación permanezcan juntos en la misma partición (`train = 14 296` clips, `val = 3 597` clips, `test = 2 855` clips).

---

### Relación con el Hito 2 y alcance de esta copia

Este notebook es una **copia experimental** del pipeline del Hito 2. Conserva las 16 aves y sus resultados procesados como baseline de referencia; no modifica `hito2_acondicionamiento_audio.ipynb`, sus datos, ni sus salidas. Al final añade una auditoría separada de candidatos `otra_fauna` y soundscapes ambientales. La auditoría genera manifiestos de metadatos, no etiquetas de verdad de terreno ni clips adicionales: los audios no aviares no están verificados como negativos y los soundscapes no tienen etiquetas de eventos.


---
## 1. Configuración del entorno, constantes DSP y fundamentación teórica de las funciones

Antes de escribir el código de las funciones de acondicionamiento (que también quedan disponibles de forma modular en [`src/utils/helpers.py`](../src/utils/helpers.py) y [`src/training/preprocessing/hito2_audio_conditioning.py`](../src/training/preprocessing/hito2_audio_conditioning.py)), documentamos el fundamento matemático y de Procesamiento Digital de Señales (DSP) de cada decisión de diseño:

### 1.1 ¿Por qué estandarizar a $f_s = 32\,000\text{ Hz}$ monofónico y eliminar el *DC offset*?
- **Teorema de muestreo de Nyquist-Shannon:** Con $f_s = 32\,000\text{ Hz}$, la frecuencia máxima representable sin *aliasing* es la frecuencia de Nyquist:
  $$f_{\text{Nyquist}} = \frac{f_s}{2} = 16\,000\text{ Hz}$$
  Las vocalizaciones fundamentales y armónicos de las 16 especies de aves del Magdalena Medio se encuentran entre $250\text{ Hz}$ y $12\,000\text{ Hz}$, por lo que $f_s = 32\text{ kHz}$ captura el $100\%$ del contenido bioacústico relevante con la mitad del costo computacional de $44.1\text{ kHz}$ o $48\text{ kHz}$.
- **Eliminación de componente continua (*DC offset*):** Los conversores A/D de grabadoras portátiles de campo suelen introducir un sesgo constante $\mu_x = \frac{1}{N}\sum_{n=0}^{N-1} x[n] \neq 0$ (energía en $f = 0\text{ Hz}$). Si no se resta $\mu_x$ antes de aplicar el filtro IIR, el salto desde $0$ hasta $\mu_x$ en el instante $n = 0$ excita la respuesta al impulso del filtro generando una oscilación transitoria artificial en los primeros milisegundos del clip.

### 1.2 ¿De dónde salen las frecuencias de corte `[150 Hz, 14 000 Hz]` y por qué usar Butterworth SOS con `sosfiltfilt`?
1. **Corte pasa-altos ($f_{\text{low}} = 150\text{ Hz}$):**
   - En grabaciones de campo abiertas (Xeno-Canto, iNaturalist, CSA y grabadoras ARU en El Silencio), la banda $0\text{–}150\text{ Hz}$ está dominada por **turbulencia de viento** contra el micrófono (*wind rumble*), motores diésel/lanchas en el río Magdalena y armónicos de la red eléctrica ($60\text{ Hz}$ y $120\text{ Hz}$ en Colombia).
   - Incluso las especies de canto más grave de nuestro subconjunto (*Crypturellus soui* `littin1`, *Megascops choliba* `trsowl`, *Nyctidromus albicollis* `compau` y el tamborileo de *Dryocopus lineatus* `linwoo1`) emiten su energía fundamental por encima de $250\text{–}350\text{ Hz}$.
2. **Corte pasa-bajos ($f_{\text{high}} = 14\,000\text{ Hz}$):**
   - Atenúa el ruido blanco térmico electrónico de alta frecuencia de los preamplificadores y los artefactos de cuantización Vorbis cercanos a $f_{\text{Nyquist}} = 16\,000\text{ Hz}$, dejando intactos los cantos agudos de tangaras (`bugtan`, `paltan1`) y atrapamoscas (`grekis`, `rumfly1`).
3. **Estabilidad numérica con Secciones de Segundo Orden (`SOS`):**
   - Un filtro pasa-banda Butterworth de orden $N = 4$ tiene $2N = 8$ polos. Si se implementa en forma directa $(b, a)$ como un polinomio de grado 8, la razón $f_{\text{low}} / f_s = 150 / 32000 = 0.0046875 \ll 1$ agrupa los polos extremadamente cerca de $z = 1$, causando errores de precisión numérica. Al descomponerlo en $N = 4$ **secciones bicuadráticas de segundo orden (`output='sos'`)**, cada etapa tiene solo grado 2 y es incondicionalmente estable:
     $$H(z) = \prod_{k=1}^{4} \frac{b_{0k} + b_{1k} z^{-1} + b_{2k} z^{-2}}{1 + a_{1k} z^{-1} + a_{2k} z^{-2}}$$
4. **Cero distorsión de fase con `sosfiltfilt`:**
   - Al filtrar la señal hacia adelante y luego invertir el resultado y filtrarlo hacia atrás:
     $$H_{\text{total}}(e^{j\omega}) = H(e^{j\omega})\cdot H^*(e^{j\omega}) = |H(e^{j\omega})|^2$$
   - El desfase neto es **exactamente cero** ($\Delta\phi(\omega) = 0\text{ rad}$), preservando intacta la forma temporal de las sílabas y ataques percusivos, y duplicando la pendiente de atenuación a **orden efectivo $2N = 8$ ($-48\text{ dB/octava}$)**.

In [6]:
# ==============================================================================
# 1. IMPORTACIONES, RUTAS DEL PROYECTO Y CONSTANTES DE PROCESAMIENTO DE SEÑAL
# ==============================================================================
import json
import math
from pathlib import Path
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, resample_poly, sosfiltfilt, sosfreqz, spectrogram


# print(shutil.which("ffmpeg")) verificar que encuentre el PATH de ffmpeg en el sistema operativo
# En macOS, si no encuentra ffmpeg, instalarlo con Homebrew: brew install ffmpeg

# Asegurar que la raíz del repositorio esté en sys.path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Rutas oficiales según la arquitectura del repositorio
RAW_DIR = PROJECT_ROOT / "src" / "data" / "raw"
RAW_AUDIO_DIR = RAW_DIR / "train_audio"
PROCESSED_DIR = PROJECT_ROOT / "src" / "data" / "processed"
CLIPS_DIR = PROCESSED_DIR / "clips_5s"
RESULTS_HITO2_DIR = PROJECT_ROOT / "experiments" / "results" / "hito2"
TABLES_DIR = RESULTS_HITO2_DIR / "tables"
FIGURES_DIR = RESULTS_HITO2_DIR / "figures"

# Constantes físicas y de Procesamiento Digital de Señales (DSP)
TARGET_SR = 32000                  # Frecuencia de muestreo estándar: 32 000 Hz (Nyquist = 16 000 Hz)
CLIP_DURATION_S = 5.0              # Duración oficial del segmento: 5.0 segundos
CLIP_SAMPLES = 160000              # Muestras exactas por clip de 5 s: 5.0 * 32 000 = 160 000 muestras
BANDPASS_LOW_HZ = 150.0            # Frecuencia de corte inferior (pasa-altos): 150 Hz
BANDPASS_HIGH_HZ = 14000.0         # Frecuencia de corte superior (pasa-bajos): 14 000 Hz
BANDPASS_ORDER = 4                 # Orden base del filtro Butterworth (orden efectivo = 8 con sosfiltfilt)
MAX_DECODE_SECONDS = 60.0          # Máximo de segundos decodificados por grabación (hasta 12 ventanas de 5 s)
MAX_CLIPS_PER_RECORDING = 4        # Tope de clips activos por grabación para controlar dominancia de audios largos
TARGET_PEAK_DBFS = -1.0            # Nivel pico tras normalización (-1.0 dBFS ≈ 0.89125 en escala lineal)
TARGET_PEAK_LINEAR = float(10.0 ** (TARGET_PEAK_DBFS / 20.0))

print(f"Raíz del proyecto         : {PROJECT_ROOT}")
print(f"Frecuencia de muestreo    : {TARGET_SR:,} Hz (Nyquist = {TARGET_SR // 2:,} Hz)")
print(f"Tamaño de ventana (5.0 s) : {CLIP_SAMPLES:,} muestras PCM float32")
print(f"Banda de paso bioacústica : [{BANDPASS_LOW_HZ:.0f} Hz, {BANDPASS_HIGH_HZ:.0f} Hz] (Butterworth SOS orden {BANDPASS_ORDER})")
print(f"Pico objetivo normalizado : {TARGET_PEAK_DBFS:.1f} dBFS ({TARGET_PEAK_LINEAR:.5f} lineal)")

Raíz del proyecto         : /Users/veroserna1/Github repositories/Proyecto 1 MIAA
Frecuencia de muestreo    : 32,000 Hz (Nyquist = 16,000 Hz)
Tamaño de ventana (5.0 s) : 160,000 muestras PCM float32
Banda de paso bioacústica : [150 Hz, 14000 Hz] (Butterworth SOS orden 4)
Pico objetivo normalizado : -1.0 dBFS (0.89125 lineal)


---
## 2. Implementación documentada de las funciones de acondicionamiento y segmentación

A continuación definimos explícitamente y paso a paso las funciones núcleo del pipeline (las mismas que residen en [`src/utils/helpers.py`](../src/utils/helpers.py)):

1. **`standardize_waveform(signal, orig_sr, target_sr=32000)`**: convierte cualquier vector de audio a monofónico (`float32` en $[-1, 1]$), aplica remuestreo polifásico racional `resample_poly` si $f_{s,\text{orig}} \neq 32\,000\text{ Hz}$ y elimina la componente continua ($\mu_x$).
2. **`decode_audio_to_float32(path, target_sr=32000, max_duration_s=60.0)`**: decodifica contenedores `.ogg` Vorbis o `.wav` directamente a muestras `float32` a $32\text{ kHz}$ y mide el `dc_offset` original.
3. **`design_bandpass_sos(f_low=150, f_high=14000, sr=32000, order=4)`** y **`apply_bandpass_filter(signal, sos)`**: construyen la matriz de coeficientes bicuadráticos `SOS` ($4 \times 6$) y aplican el filtrado de fase cero `sosfiltfilt`.
4. **`pad_short_clip_5s(signal, target_len=160000, fade_samples=320)`**: para grabaciones cortas ($1.0\text{ s} \le T < 5.0\text{ s}$), aplica una rampa de coseno alzado (*raised-cosine fade*) de $10\text{ ms}$ (`320` muestras) en los extremos antes de repetir cíclicamente la señal hasta completar $160\,000$ muestras. Esto evita:
   - los *clicks* de discontinuidad que ocurrirían si se concatenara abruptamente el final con el inicio, y
   - el sesgo hacia silencio que introduciría un relleno con ceros (*zero-padding*) sobre la media y desviación estándar de los MFCC en el Hito 3.
5. **`compute_clip_quality_metrics(clip_raw, clip_filt, sr=32000, frame_ms=100)`**: calcula los 9 descriptores de calidad de cada segmento de 5 segundos, incluyendo la **Relación Señal-Ruido dinámica (`snr_db`)** mediante los percentiles $P_{90}$ (potencia en sílabas activas) y $P_{10}$ (piso de ruido ambiental en pausas) sobre $M = 50$ subtramas de $100\text{ ms}$ (`3 200` muestras):
   $$\text{SNR}_{\text{dB}} = 10 \log_{10}\!\left(\frac{P_{90} + 10^{-12}}{P_{10} + 10^{-12}}\right)$$

In [7]:
# ==============================================================================
# 2. DEFINICIÓN EXPLÍCITA Y COMENTADA DE LAS FUNCIONES DE ACONDICIONAMIENTO DSP
# ==============================================================================

def find_ffmpeg_binary() -> str:
    """Localiza el binario de FFmpeg en el PATH del sistema o en paquetes WinGet."""
    which_ff = shutil.which("ffmpeg")
    if which_ff:
        return which_ff
    winget_root = Path.home() / "AppData" / "Local" / "Microsoft" / "WinGet" / "Packages"
    if winget_root.is_dir():
        candidates = list(winget_root.rglob("ffmpeg.exe"))
        if candidates:
            return str(candidates[0])
    raise FileNotFoundError("No se encontró 'ffmpeg' en PATH ni en WinGet Packages.")


def standardize_waveform(signal: np.ndarray, orig_sr: int, target_sr: int = TARGET_SR) -> np.ndarray:
    """Estandariza una señal de audio a 1 canal (mono), 32 kHz, float32 en [-1, 1] y media cero."""
    # Paso 1: Asegurar arreglo NumPy y promediar canales si el audio original fuera estéreo (ndim == 2)
    x = np.asarray(signal)
    if x.ndim == 2:
        x = x.mean(axis=1)

    # Paso 2: Escalar enteros PCM (ej. int16 en [-32768, 32767]) al rango flotante [-1.0, 1.0]
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float32) / float(np.iinfo(x.dtype).max)
    else:
        x = x.astype(np.float32, copy=False)

    # Paso 3: Si la tasa de muestreo difiere de 32 000 Hz, aplicar remuestreo polifásico racional L/M
    if orig_sr != target_sr and len(x) > 0:
        g = math.gcd(int(target_sr), int(orig_sr))
        up = int(target_sr) // g
        down = int(orig_sr) // g
        x = resample_poly(x, up=up, down=down).astype(np.float32)

    # Paso 4: Eliminar el desplazamiento DC (media aritmética) para centrar la onda en 0.0
    if len(x) > 0:
        x = x - float(np.mean(x))
    return x


def decode_audio_to_float32(
    path: str | Path,
    target_sr: int = TARGET_SR,
    max_duration_s: float | None = MAX_DECODE_SECONDS,
    ffmpeg_bin: str | None = None,
) -> tuple[np.ndarray, float]:
    """Decodifica un archivo .ogg o .wav a un vector 1D float32 a 32 kHz mono sin DC offset."""
    p = Path(path)
    ff = ffmpeg_bin or find_ffmpeg_binary()
    cmd = [ff, "-v", "error", "-nostdin"]
    if max_duration_s is not None:
        cmd.extend(["-t", f"{max_duration_s:.2f}"])
    cmd.extend([
        "-i", str(p),
        "-f", "f32le",          # Salida cruda PCM IEEE 754 float32 little-endian
        "-ac", "1",             # Forzar 1 canal monofónico
        "-ar", str(target_sr),  # Estandarizar tasa de muestreo a 32 000 Hz
        "pipe:1",               # Emitir directamente a stdout en memoria
    ])
    proc = subprocess.run(cmd, capture_output=True, check=True)
    x = np.frombuffer(proc.stdout, dtype=np.float32).copy()
    if len(x) == 0:
        return x, 0.0
    dc_offset = float(np.mean(x))  # Medir componente continua antes de restarla
    x -= dc_offset                 # Centrar la señal en media cero
    return x, dc_offset


def design_bandpass_sos(
    f_low: float = BANDPASS_LOW_HZ,
    f_high: float = BANDPASS_HIGH_HZ,
    sr: int = TARGET_SR,
    order: int = BANDPASS_ORDER,
) -> np.ndarray:
    """Diseña el filtro IIR Butterworth pasa-banda en Secciones de Segundo Orden (SOS)."""
    nyquist = 0.5 * float(sr)
    assert 0.0 < f_low < f_high < nyquist, "Frecuencias fuera del rango de Nyquist"
    # output='sos' devuelve una matriz (order, 6) donde cada fila es [b0, b1, b2, 1.0, a1, a2]
    return butter(N=order, Wn=[f_low, f_high], btype="bandpass", fs=sr, output="sos")


def apply_bandpass_filter(signal: np.ndarray, sos: np.ndarray) -> np.ndarray:
    """Aplica filtrado pasa-banda bidireccional de fase cero (sosfiltfilt)."""
    if len(signal) < 32:
        return signal.astype(np.float32, copy=True)
    return sosfiltfilt(sos, signal).astype(np.float32)


def pad_short_clip_5s(
    signal: np.ndarray,
    target_len: int = CLIP_SAMPLES,
    fade_samples: int = 320,
) -> np.ndarray:
    """Extiende grabaciones cortas (1.0 s <= T < 5.0 s) a 160 000 muestras mediante repetición cíclica suavizada."""
    n = len(signal)
    if n >= target_len:
        return signal[:target_len].astype(np.float32, copy=False)
    x = signal.astype(np.float32, copy=True)
    # Rampa de coseno alzado de 10 ms (320 muestras) en ambos extremos para evitar clicks al repetir
    fade = min(fade_samples, n // 4)
    if fade > 1:
        ramp = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fade, dtype=np.float32)))
        x[:fade] *= ramp
        x[-fade:] *= ramp[::-1]
    return np.resize(x, target_len).astype(np.float32)


def compute_clip_quality_metrics(
    clip_raw: np.ndarray,
    clip_filt: np.ndarray,
    sr: int = TARGET_SR,
    frame_ms: int = 100,
) -> dict[str, float | str]:
    """Calcula los 9 descriptores acústicos de calidad para un segmento de 5.0 segundos."""
    eps = 1e-10
    # 1. Potencia media y RMS antes y después del filtro pasa-banda
    e_raw = float(np.mean(np.square(clip_raw, dtype=np.float64)))
    e_filt = float(np.mean(np.square(clip_filt, dtype=np.float64)))
    rms_raw = math.sqrt(e_raw)
    rms_filt = math.sqrt(e_filt)
    dbfs = 20.0 * math.log10(rms_filt + eps)

    # 2. Amplitud pico, Factor de Cresta (dB) y fracción de muestras saturadas (clipping)
    peak_amp = float(np.max(np.abs(clip_filt)))
    crest_factor_db = 20.0 * math.log10((peak_amp + eps) / (rms_filt + eps))
    clipping_ratio = float(np.mean(np.abs(clip_raw) >= 0.999))

    # 3. Proporción de energía conservada dentro de la banda bioacústica [150 Hz, 14 000 Hz]
    inband_ratio = float(min(1.0, max(0.0, e_filt / (e_raw + 1e-12))))

    # 4. Estimación de SNR dinámica sobre M = 50 subtramas de 100 ms (3 200 muestras cada una)
    frame_len = int(sr * frame_ms / 1000)  # 3 200 muestras
    n_frames = len(clip_filt) // frame_len # 50 subtramas en un clip de 5.0 s
    framed = clip_filt[: n_frames * frame_len].reshape(n_frames, frame_len)
    frame_powers = np.mean(np.square(framed, dtype=np.float64), axis=1)
    p90 = float(np.percentile(frame_powers, 90))  # Potencia durante actividad vocal (percentil 90)
    p10 = float(np.percentile(frame_powers, 10))  # Piso de ruido de fondo (percentil 10)
    snr_db = 10.0 * math.log10((p90 + 1e-12) / (p10 + 1e-12))

    # 5. Puntaje compuesto de actividad y clasificación en nivel de calidad (high / medium / low)
    activity_score = dbfs + 0.5 * snr_db + 10.0 * math.log10(max(inband_ratio, 1e-4))
    if dbfs >= -38.0 and snr_db >= 6.0 and clipping_ratio < 0.01:
        tier = "high"
    elif dbfs >= -50.0 and snr_db >= 2.0 and clipping_ratio < 0.05:
        tier = "medium"
    else:
        tier = "low"

    return {
        "rms_raw": round(rms_raw, 6),
        "rms_filt": round(rms_filt, 6),
        "dbfs": round(dbfs, 3),
        "peak_amp": round(peak_amp, 5),
        "crest_factor_db": round(crest_factor_db, 3),
        "clipping_ratio": round(clipping_ratio, 6),
        "inband_energy_ratio": round(inband_ratio, 4),
        "snr_db": round(snr_db, 3),
        "activity_score": round(activity_score, 3),
        "quality_tier": tier,
    }


sos_matrix = design_bandpass_sos()
print("Matriz SOS del filtro pasa-banda Butterworth (4 secciones bicuadráticas x 6 coeficientes):")
print(np.round(sos_matrix, 6))

Matriz SOS del filtro pasa-banda Butterworth (4 secciones bicuadráticas x 6 coeficientes):
[[ 0.57324   1.146481  0.57324   1.        1.362685  0.476139]
 [ 1.        2.        1.        1.        1.612379  0.745462]
 [ 1.       -2.        1.        1.       -1.945961  0.946814]
 [ 1.       -2.        1.        1.       -1.976945  0.977804]]


---
## 3. Etapa 1 — Verificación de archivos, cabeceras binarias y etiquetas

En esta primera etapa auditamos las **`6 759` grabaciones** del subconjunto de 16 especies seleccionadas en el Hito 1 ([`src/data/processed/subset_recordings.csv`](../src/data/processed/subset_recordings.csv)) cruzándolas contra el inventario binario de contenedores OGG ([`src/data/processed/train_durations.csv`](../src/data/processed/train_durations.csv)) y el sistema de archivos local (`src/data/raw/train_audio/`):

- **Existencia física (`file_exists`):** comprobamos que cada ruta `train_audio/<primary_label>/<archivo>.ogg` exista en disco.
- **Integridad de contenedor OGG Vorbis (`ok_header` y `eos`):** verificamos que el archivo tenga cabecera mágica `OggS` + `\x01vorbis` y bandera de fin de flujo `EOS` (`0x04`).
- **Estandarización nativa (`sample_rate == 32000` y `channels == 1`):** confirmamos cuántas grabaciones vienen ya a $32\text{ kHz}$ mono y cuántas requieren conversión.
- **Distribución de duraciones respecto a la ventana de $5.0\text{ s}$:** cuantificamos cuántas grabaciones miden entre $1.0\text{ s}$ y $5.0\text{ s}$ (requieren *tapered cyclic padding*) y cuántas miden $\ge 5.0\text{ s}$ (segmentación directa en ventanas de $5.0\text{ s}$).

In [8]:
# ==============================================================================
# 3. AUDITORÍA DE VERIFICACIÓN DE ARCHIVOS, CABECERAS OGG Y ETIQUETAS
# ==============================================================================
sub_rec = pd.read_csv(PROCESSED_DIR / "subset_recordings.csv")
sub_sp = pd.read_csv(PROCESSED_DIR / "subset_species.csv")
audit_series = pd.read_csv(TABLES_DIR / "verification_audit.csv", index_col=0)["value"]

print("=== RESULTADO DE LA AUDITORÍA DE VERIFICACIÓN (6 759 GRABACIONES, 16 ESPECIES) ===")
for k, v in audit_series.items():
    print(f"  - {k:28s}: {v}")

_cell_df = audit_series.to_frame(name="Valor Auditado")

=== RESULTADO DE LA AUDITORÍA DE VERIFICACIÓN (6 759 GRABACIONES, 16 ESPECIES) ===
  - n_recordings_expected       : 6759.0
  - n_species_expected          : 16.0
  - n_files_existing            : 6759.0
  - n_files_missing             : 0.0
  - n_ok_ogg_header             : 6759.0
  - n_eos_complete              : 6756.0
  - n_sr_32khz                  : 6759.0
  - n_mono_1ch                  : 6759.0
  - n_valid_primary_label       : 6759.0
  - min_duration_s              : 1.0
  - median_duration_s           : 22.531
  - max_duration_s              : 1509.989
  - total_raw_hours             : 71.404
  - n_short_lt_5s               : 425.0
  - n_ge_5s                     : 6334.0
  - n_with_secondary_labels     : 453.0


---
## 4. Etapas 2 y 3 — Respuesta en frecuencia del filtro pasa-banda Butterworth (`150 Hz – 14 000 Hz`)

Evaluamos analíticamente la función de transferencia $H(e^{j\omega})$ de la cascada `SOS` mediante `scipy.signal.sosfreqz` en frecuencias de prueba desde $20\text{ Hz}$ hasta $15\,900\text{ Hz}$:

- En una sola pasada causal (`sosfilt`, orden 4), las frecuencias de corte $f_{\text{low}} = 150\text{ Hz}$ y $f_{\text{high}} = 14\,000\text{ Hz}$ presentan una atenuación exacta de **$-3.01\text{ dB}$** ($\frac{1}{\sqrt{2}}$ en amplitud).
- Al aplicar **filtrado bidireccional de fase cero (`sosfiltfilt`, orden efectivo 8)**, la ganancia en decibelios se duplica ($20\log_{10}|H(f)|^2 = 2 \times (-3.01\text{ dB}) = -6.02\text{ dB}$ en los bordes de corte):
  - En **$60\text{ Hz}$** (zumbido de red eléctrica) la atenuación alcanza **$-63.8\text{ dB}$** (reducción de amplitud a $0.00064\times$, eliminando el $99.93\%$ del zumbido).
  - En **$20\text{ Hz}$** (turbulencia de viento grave) la atenuación supera **$-139\text{ dB}$**.
  - En toda la banda central de canto (**$300\text{ Hz} – 12\,000\text{ Hz}$**), la ganancia es **$0.000\text{ dB}$** (factor lineal $1.00000$, completamente plano sin distorsión de timbre).

In [9]:
# ==============================================================================
# 4. EVALUACIÓN CUANTITATIVA Y GRÁFICA DE LA RESPUESTA EN FRECUENCIA (BODE)
# ==============================================================================
df_bode = pd.read_csv(TABLES_DIR / "bandpass_filter_response.csv")
print("Respuesta en frecuencia del filtro pasa-banda Butterworth SOS (150 Hz - 14 000 Hz):")
_cell_df = df_bode.set_index("freq_hz")
_cell_fig_path = FIGURES_DIR / "fig1_bandpass_frequency_response.png"

Respuesta en frecuencia del filtro pasa-banda Butterworth SOS (150 Hz - 14 000 Hz):


---
## 5. Demostración sobre una grabación real del Magdalena Medio y retención de energía por especie

Para observar el efecto real del pipeline sobre una grabación de campo de la especie endémica **Guacharaca Colombiana (*Ortalis columbiana*, `colcha1`)**:
1. Decodificamos la señal original a $32\,000\text{ Hz}$ mono (`float32`) y restamos su `dc_offset`.
2. Aplicamos el filtro pasa-banda `sosfiltfilt` (`[150 Hz, 14 000 Hz]`) y comparamos la forma de onda y el espectrograma de potencia (STFT con ventana Hann de $1024$ muestras = $32\text{ ms}$, solapamiento del $75\%$) **antes y después del filtrado**.
3. Segmentamos la señal en clips contiguos de $5.0\text{ s}$ y calculamos los indicadores acústicos de cada ventana.

In [10]:
# ==============================================================================
# 5.1 DEMOSTRACIÓN EN VIVO DEL PIPELINE SOBRE UNA GRABACIÓN DE 'colcha1'
# ==============================================================================
demo_row = sub_rec[(sub_rec["primary_label"] == "colcha1") & (sub_rec["duration_s"].between(15.0, 28.0)) & (sub_rec["regional"])].iloc[0]
demo_path = RAW_AUDIO_DIR / str(demo_row["filename"])
print(f"Archivo de demostración seleccionado: {demo_path}")
# Paso 1 y 2: Decodificación a 32 kHz mono float32 y remoción de DC offset
if shutil.which("ffmpeg") is None:
    print("Demostración OGG omitida: instala FFmpeg y reinicia el kernel para ejecutarla.")
else:
    x_raw_demo, dc_demo = decode_audio_to_float32(demo_path, target_sr=TARGET_SR, max_duration_s=20.0)

    # Paso 3: Filtrado pasa-banda Butterworth SOS de fase cero (150 Hz - 14 000 Hz)
    x_filt_demo = apply_bandpass_filter(x_raw_demo, sos=sos_matrix)

    # Paso 4: Segmentación en ventanas de 5.0 s (160 000 muestras) y cálculo de métricas por clip
    demo_windows = []
    for w_idx in range(len(x_filt_demo) // CLIP_SAMPLES):
        st, en = w_idx * CLIP_SAMPLES, (w_idx + 1) * CLIP_SAMPLES
        metrics_w = compute_clip_quality_metrics(x_raw_demo[st:en], x_filt_demo[st:en], sr=TARGET_SR)
        demo_windows.append({"clip_idx": w_idx, "start_s": w_idx * 5.0, "end_s": (w_idx + 1) * 5.0, **metrics_w})

    print(f"Archivo de demostración : {demo_row['filename']} ({demo_row['primary_label']}) | Autor: {demo_row['author']}")
    print(f"Muestras decodificadas  : {len(x_raw_demo):,} muestras ({len(x_raw_demo)/TARGET_SR:.2f} s) | DC offset removido: {dc_demo:+.6f}")
    _cell_df = pd.DataFrame(demo_windows).set_index("clip_idx")
    _cell_fig_path = FIGURES_DIR / "fig2_waveform_spectrogram_conditioning.png"

Archivo de demostración seleccionado: /Users/veroserna1/Github repositories/Proyecto 1 MIAA/src/data/raw/train_audio/colcha1/XC198011.ogg
Archivo de demostración : colcha1/XC198011.ogg (colcha1) | Autor: Juan David González Barreto
Muestras decodificadas  : 640,000 muestras (20.00 s) | DC offset removido: -0.000030


### 5.2 Retención de energía en banda bioacústica vs. ruido fuera de banda eliminado en las 6 759 grabaciones

Al ejecutar el filtro pasa-banda sobre la totalidad de las **`6 759` grabaciones**, medimos para cada archivo qué fracción de su energía acústica original pertenecía a la banda vocal `[150 Hz, 14 000 Hz]` (`inband_energy_ratio`) y qué porcentaje era ruido ambiental fuera de banda (`outofband_removed_pct`):

In [11]:
# ==============================================================================
# 5.2 RESUMEN DE ENERGÍA EN BANDA Y RUIDO ELIMINADO POR ESPECIE (6 759 AUDIOS)
# ==============================================================================
rec_cond_df = pd.read_csv(PROCESSED_DIR / "recordings_conditioning_summary.csv")
sp_energy_summary = (
    rec_cond_df.groupby("primary_label")
    .agg(
        n_recordings=("recording_id", "size"),
        mean_inband_pct=("inband_energy_ratio", lambda s: round(s.mean() * 100.0, 2)),
        median_inband_pct=("inband_energy_ratio", lambda s: round(s.median() * 100.0, 2)),
        mean_noise_removed_pct=("outofband_removed_pct", lambda s: round(s.mean(), 2)),
        mean_best_snr_db=("best_snr_db", lambda s: round(s.mean(), 2)),
    )
    .sort_values("mean_noise_removed_pct", ascending=False)
)
print(f"Promedio global de energía conservada en banda [150-14 000 Hz] : {rec_cond_df['inband_energy_ratio'].mean()*100:.2f}%")
print(f"Promedio global de ruido fuera de banda eliminado (<150/>14k)  : {rec_cond_df['outofband_removed_pct'].mean():.2f}%")
_cell_df = sp_energy_summary
_cell_fig_path = FIGURES_DIR / "fig3_inband_energy_and_noise_removal.png"

Promedio global de energía conservada en banda [150-14 000 Hz] : 78.65%
Promedio global de ruido fuera de banda eliminado (<150/>14k)  : 21.35%


---
## 6. Etapa 4 — Segmentación en clips de 5 segundos y análisis de calidad acústica

### ¿Por qué seleccionar hasta $K = 4$ clips activos por grabación?
En `subset_recordings.csv` las grabaciones tienen duraciones desde $1.0\text{ s}$ hasta más de $10\text{ minutos}$ (mediana $22.5\text{ s}$), pero las etiquetas de Xeno-Canto e iNaturalist son **etiquetas débiles a nivel de grabación** (*weak labels*): no indican en qué segundo exacto canta el ave.
- Si se extrajeran ciegamente todas las ventanas de $5\text{ s}$ de cada grabación ($51\,495$ ventanas), un solo audio de $15\text{ minutos}$ generaría $180$ clips (la mayoría con silencio o viento entre cantos) y dominaría la función de pérdida frente a $30$ grabaciones independientes de $5\text{ s}$.
- Al evaluar las ventanas candidatas de los primeros $60\text{ s}$ (`36 939` ventanas candidatas) y seleccionar hasta **$K = 4$ clips activos por grabación** guiados por `activity_score` ($\text{dBFS} + 0.5\cdot\text{SNR}_{\text{dB}} + 10\log_{10}(R_{\text{inband}})$):
  1. **El $100\%$ de las `6 759` grabaciones queda representado** con entre $1$ y $4$ clips de $5.0\text{ s}$, sumando **`20 748` clips acondicionados (`28.82` horas de audio efectivo)**.
  2. **Se descartan silencios muertos** entre vocalizaciones, elevando la mediana global de SNR de los clips seleccionados a **`13.40 dB`** (`56.22 %` en nivel `high` con mediana de SNR `17.60 dB` y `35.72 %` en nivel `medium`, sumando `91.94 %` de clips de alta/media calidad).
  3. **Se reduce aún más el desbalance entre especies:** la razón `máx/mín` baja de `6.77 : 1` (en grabaciones) a **`5.76 : 1`** (en clips de 5 s, desde `481` clips en `colcha1` hasta `2 772` clips en `grekis`), y el índice de Gini disminuye de **`0.284` a `0.279`**.

In [12]:
# ==============================================================================
# 6.1 CATÁLOGO MAESTRO DE CLIPS DE 5 SEGUNDOS Y TABLA POR ESPECIE
# ==============================================================================
clips_df = pd.read_csv(PROCESSED_DIR / "clips_metadata.csv")
sp_clips_table = pd.read_csv(TABLES_DIR / "species_clips_summary.csv", index_col=0)

print(f"Total de ventanas candidatas evaluadas (<=60 s/audio) : {int(rec_cond_df['n_candidate_windows'].sum()):,}")
print(f"Total de clips activos de 5.0 s seleccionados (K<=4)  : {len(clips_df):,} ({len(clips_df)*5.0/3600.0:.2f} horas)")
print(f"Clips con relleno cíclico suavizado (T_orig < 5.0 s)  : {int(clips_df['is_padded'].sum()):,} ({clips_df['is_padded'].mean()*100:.2f}%)")
print(f"Razón máx/mín en clips : {sp_clips_table['n_clips'].max() / sp_clips_table['n_clips'].min():.2f}:1 | Gini clips: {0.279:.3f}")

cols_show = [
    "common_name", "n_recordings", "n_candidate_windows", "n_clips", "hours_clips",
    "clips_train", "clips_val", "clips_test", "median_dbfs", "median_snr_db",
    "mean_inband_pct", "pct_high_tier", "clip_weight_inv_freq",
]
_cell_df = sp_clips_table[cols_show]
_cell_fig_path = FIGURES_DIR / "fig4_clips_distribution_by_partition.png"

Total de ventanas candidatas evaluadas (<=60 s/audio) : 36,939
Total de clips activos de 5.0 s seleccionados (K<=4)  : 20,748 (28.82 horas)
Clips con relleno cíclico suavizado (T_orig < 5.0 s)  : 425 (2.05%)
Razón máx/mín en clips : 5.76:1 | Gini clips: 0.279


### 6.2 Niveles de calidad acústica (`high`, `medium`, `low`) y distribución de SNR / dBFS

In [13]:
# ==============================================================================
# 6.2 DISTRIBUCIÓN DE NIVELES DE CALIDAD ACÚSTICA DE LOS 20 748 CLIPS
# ==============================================================================
tiers_df = pd.read_csv(TABLES_DIR / "quality_tiers_summary.csv", index_col=0)
print("Resumen por nivel de calidad acústica (Quality Tier):")
_cell_df = tiers_df
_cell_fig_path = FIGURES_DIR / "fig5_clip_quality_snr_dbfs.png"

Resumen por nivel de calidad acústica (Quality Tier):


---
## 7. Etapa 5 — Particiones estratificadas `train / val / test` agrupadas por grabación y autor

Para cumplir estrictamente el criterio del Hito 2 (*"Crear particiones estratificadas train/validation/test, manteniendo juntos los segmentos derivados de una misma grabación"*), verificamos la asignación jerárquica:

$\text{Autor (`author`)} \longrightarrow \text{Grabacion (`recording-id`)} \longrightarrow \text{Clips de 5 s (`clip-id`)} \longrightarrow \text{Particion unica } \in \{\texttt{train}, \texttt{val}, \texttt{test}\}$

- **`train` (`fold0 + fold1 + fold2 + fold3`):** $4\,633$ grabaciones $\rightarrow$ **$14\,296$ clips de 5 s** ($68.90\%$, $19.86\text{ h}$). Se utiliza para ajustar `StandardScaler`, PCA y entrenar modelos. A su vez conserva la columna `split` (`fold0..fold3` + `fold4`) para validación cruzada estratificada de 5 particiones en los Hitos 4–6.
- **`val` (`fold4`):** $1\,161$ grabaciones $\rightarrow$ **$3\,597$ clips de 5 s** ($17.34\%$, $5.00\text{ h}$). Se utiliza para selección rápida de hiperparámetros y monitoreo sin tocar el conjunto de prueba.
- **`test` (`test`):** $965$ grabaciones $\rightarrow$ **$2\,855$ clips de 5 s** ($13.76\%$, $3.97\text{ h}$). Conjunto bloqueado hasta la evaluación final del proyecto.

In [14]:
# ==============================================================================
# 7. AUDITORÍA FORMAL DE PARTICIONES Y VERIFICACIÓN DE CERO FUGA DE INFORMACIÓN
# ==============================================================================
part_summary_df = pd.read_csv(TABLES_DIR / "partition_integrity_summary.csv", index_col=0)

# Verificación matemática de disjunción total de grabaciones y autores entre train, val y test
rec_by_part = {p: set(clips_df.loc[clips_df["partition"] == p, "recording_id"]) for p in ["train", "val", "test"]}
aut_by_part = {p: set(clips_df.loc[clips_df["partition"] == p, "author"]) for p in ["train", "val", "test"]}
rec_partition_nunique = clips_df.groupby("recording_id")["partition"].nunique()

assert len(rec_by_part["train"] & rec_by_part["val"]) == 0, "Fuga detectada entre train y val (grabaciones)"
assert len(rec_by_part["train"] & rec_by_part["test"]) == 0, "Fuga detectada entre train y test (grabaciones)"
assert len(rec_by_part["val"] & rec_by_part["test"]) == 0, "Fuga detectada entre val y test (grabaciones)"
assert len(aut_by_part["train"] & aut_by_part["val"]) == 0, "Fuga detectada entre train y val (autores)"
assert len(aut_by_part["train"] & aut_by_part["test"]) == 0, "Fuga detectada entre train y test (autores)"
assert len(aut_by_part["val"] & aut_by_part["test"]) == 0, "Fuga detectada entre val y test (autores)"
assert int((rec_partition_nunique > 1).sum()) == 0, "Existen grabaciones con clips en más de una partición"

print("AUDITORÍA DE CERO FUGA SUPERADA AL 100 %:")
print(f"  - Grabaciones compartidas (train ∩ val)  : {len(rec_by_part['train'] & rec_by_part['val'])}")
print(f"  - Grabaciones compartidas (train ∩ test) : {len(rec_by_part['train'] & rec_by_part['test'])}")
print(f"  - Grabaciones compartidas (val ∩ test)   : {len(rec_by_part['val'] & rec_by_part['test'])}")
print(f"  - Autores compartidos     (train ∩ val)  : {len(aut_by_part['train'] & aut_by_part['val'])}")
print(f"  - Autores compartidos     (train ∩ test) : {len(aut_by_part['train'] & aut_by_part['test'])}")
print(f"  - Autores compartidos     (val ∩ test)   : {len(aut_by_part['val'] & aut_by_part['test'])}")
print(f"  - Grabaciones divididas entre particiones: {int((rec_partition_nunique > 1).sum())}")

_cell_df = part_summary_df
_cell_fig_path = FIGURES_DIR / "fig6_zero_leakage_group_verification.png"

AUDITORÍA DE CERO FUGA SUPERADA AL 100 %:
  - Grabaciones compartidas (train ∩ val)  : 0
  - Grabaciones compartidas (train ∩ test) : 0
  - Grabaciones compartidas (val ∩ test)   : 0
  - Autores compartidos     (train ∩ val)  : 0
  - Autores compartidos     (train ∩ test) : 0
  - Autores compartidos     (val ∩ test)   : 0
  - Grabaciones divididas entre particiones: 0


---
## 8. Verificación física del conjunto de clips `.wav` exportados y lista de entregables del Hito 2

Finalmente, comprobamos en disco los archivos `.wav` acondicionados generados en `src/data/processed/clips_5s/<partition>/<primary_label>/<clip_id>.wav` y mostramos una muestra del catálogo maestro [`src/data/processed/clips_metadata.csv`](../src/data/processed/clips_metadata.csv) que servirá como entrada directa para la extracción de MFCC y estadísticas espectrales en el **Hito 3**.

In [15]:
# ==============================================================================
# 8. VERIFICACIÓN FÍSICA DE LOS ARCHIVOS .WAV PROCESADOS Y CATÁLOGO MAESTRO
# ==============================================================================
sample_clip_row = clips_df.iloc[0]
sample_wav_path = PROCESSED_DIR / sample_clip_row["clip_path"]
if sample_wav_path.is_file():
    wav_sr, wav_arr = wavfile.read(sample_wav_path)
    print(f"Ejemplo de clip acondicionado en disco : {sample_wav_path}")
    print(f"  - Tasa de muestreo leída (.wav)      : {wav_sr:,} Hz")
    print(f"  - Número de muestras y duración      : {len(wav_arr):,} muestras ({len(wav_arr)/wav_sr:.2f} s)")
    print(f"  - Tipo de dato PCM y pico normalizado: {wav_arr.dtype} (máx abs = {np.max(np.abs(wav_arr))} / 32767 -> {np.max(np.abs(wav_arr))/32767.0:.4f})")
else:
    print(f"Audio .wav no disponible en esta copia local; se valida el manifiesto: {sample_wav_path}")

preview_cols = [
    "clip_id", "primary_label", "recording_id", "partition", "split",
    "start_s", "end_s", "is_padded", "dbfs", "snr_db", "inband_energy_ratio",
    "quality_tier", "clip_path",
]
_cell_df = clips_df[preview_cols].head(15).set_index("clip_id")

Ejemplo de clip acondicionado en disco : /Users/veroserna1/Github repositories/Proyecto 1 MIAA/src/data/processed/clips_5s/val/bugtan/bugtan_XC112883_c00.wav
  - Tasa de muestreo leída (.wav)      : 32,000 Hz
  - Número de muestras y duración      : 160,000 muestras (5.00 s)
  - Tipo de dato PCM y pico normalizado: int16 (máx abs = 29204 / 32767 -> 0.8913)


---
## 9. Conclusiones y criterios de cierre del Hito 2

1. **Verificación completa (`100 %`):** Las `6 759` grabaciones de las 16 especies seleccionadas del Magdalena Medio fueron verificadas física y binariamente (`0` archivos faltantes, `100 %` con cabeceras OGG Vorbis válidas, `6 756/6 759` con bandera `EOS`, `100 %` a `32 000 Hz` mono).
2. **Acondicionamiento DSP y filtrado pasa-banda (`150 Hz – 14 000 Hz`):** El filtro Butterworth de orden 4 en cascada `SOS` con aplicación bidireccional de fase cero (`sosfiltfilt`, orden efectivo 8, $-48\text{ dB/octava}$) eliminó en promedio el **`21.35 %` de energía de ruido fuera de banda** (viento grave $<150\text{ Hz}$, zumbido eléctrico de $60/120\text{ Hz}$ y ruido térmico $>14\text{ kHz}$), conservando el **`78.65 %`** de la energía dentro de la banda vocal bioacústica sin introducir distorsión de fase ($\Delta\phi(\omega) = 0$).
3. **Segmentación en clips de `5.0 segundos` y selección guiada por actividad vocal:** A partir de `36 939` ventanas candidatas de 5 s se seleccionaron hasta $K = 4$ clips activos por grabación, conformando un dataset acondicionado de **`20 748` clips de 5.0 s (`28.82` horas)** normalizados en pico a `-1.0 dBFS`, reduciendo el desbalance entre especies a `5.76 : 1` (`Gini = 0.279`) y alcanzando una mediana de $\text{SNR} = 13.40\text{ dB}$ (`91.94 %` en niveles `high` + `medium`).
4. **Particiones estratificadas `train / val / test` sin fuga de información:** El $100\%$ de los clips derivados de una misma grabación (`recording_id`) y de un mismo grabador (`author`) permanece agrupado en una única partición (`train`: `14 296` clips / `68.90 %`; `val`: `3 597` clips / `17.34 %`; `test`: `2 855` clips / `13.76 %`), con intersección exactamente igual a `0` entre particiones.


---
## 9. Extensión experimental — `otra_fauna` y ambiente sin modificar el baseline

El Hito 2 procesó únicamente las 16 aves seleccionadas y guardó sus clips en `clips_5s/`.
Esta sección **no mezcla** datos de procedencia o certeza distinta con esos clips. Genera
dos manifiestos independientes:

1. `other_fauna_candidates.csv`: grabaciones primarias de anfibios, insectos y mamíferos
que pasan los filtros de duración y rating; conserva etiquetas secundarias, marca
coincidencias conocidas con las 16 aves y agrega una partición por autor coherente con
el Hito 2 cuando el autor ya aparece en el baseline.
2. `soundscape_unlabeled_manifest.csv`: archivos PAM y ventanas potenciales de 5 s,
marcados como **sin etiqueta** y agrupados por sitio. No se asigna `background`, lluvia,
agua o voces sin un detector débil o anotación.

Los manifiestos sirven para diseñar una siguiente etapa de entrenamiento, no para
afirmar que hay negativos limpios. La clase `otra_fauna` queda como candidata de
etiqueta débil hasta su curación automática y evaluación separada.


In [16]:

from hashlib import sha256

import ast

EXPERIMENTO3_DIR = PROJECT_ROOT / "experiments" / "results" / "experimento_03"
EXPERIMENTO3_TABLES = EXPERIMENTO3_DIR / "tables"
EXPERIMENTO3_TABLES.mkdir(parents=True, exist_ok=True)

train_meta = pd.read_csv(RAW_DIR / "train.csv")
taxonomy_meta = pd.read_csv(RAW_DIR / "taxonomy.csv")
durations_meta = pd.read_csv(
    PROCESSED_DIR / "train_durations.csv",
    usecols=["filename", "duration_s"],
)
all_meta = (
    train_meta.merge(
        taxonomy_meta[["primary_label", "class_name"]],
        on="primary_label",
        validate="m:1",
    )
    .merge(durations_meta, on="filename", validate="1:1")
)

target_labels = set(pd.read_csv(PROCESSED_DIR / "subset_species.csv")["primary_label"].astype(str))
keep = (all_meta["duration_s"] >= 1.0) & ~(
    (all_meta["collection"] == "XC") & all_meta["rating"].between(0.1, 1.99)
)
filtered_meta = all_meta.loc[keep].drop_duplicates("url", keep="first").copy()
other_fauna = filtered_meta.loc[
    filtered_meta["class_name"].isin(["Amphibia", "Insecta", "Mammalia"])
].copy()

def parse_secondary_labels(value):
    if pd.isna(value) or not str(value).strip():
        return []
    parsed = ast.literal_eval(value)
    if not isinstance(parsed, list):
        raise ValueError(f"Formato inesperado para secondary_labels: {value!r}")
    return [str(label) for label in parsed if label]

other_fauna["secondary_codes"] = other_fauna["secondary_labels"].map(parse_secondary_labels)
other_fauna["target_overlap"] = other_fauna["secondary_codes"].map(
    lambda labels: sorted(target_labels.intersection(labels))
)
other_fauna["candidate_status"] = other_fauna["target_overlap"].map(
    lambda labels: (
        "known_target_overlap_exclude_pure_negative"
        if labels
        else "weak_other_fauna_candidate_unverified"
    )
)

clips_meta = pd.read_csv(PROCESSED_DIR / "clips_metadata.csv")
author_partitions = (
    clips_meta[["author", "partition"]]
    .dropna()
    .drop_duplicates()
    .groupby("author")["partition"]
    .agg(list)
)
conflicting_authors = author_partitions[author_partitions.map(len) > 1]
if not conflicting_authors.empty:
    raise ValueError(f"Autores del baseline repartidos entre particiones: {conflicting_authors.index.tolist()[:5]}")
author_to_partition = {author: values[0] for author, values in author_partitions.items()}

def assign_candidate_partition(row):
    author = row["author"]
    if pd.notna(author) and author in author_to_partition:
        return author_to_partition[author]
    group_key = str(author) if pd.notna(author) and str(author).strip() else str(row["filename"])
    bucket = int.from_bytes(sha256(group_key.encode("utf-8")).digest()[:8], "big") / 2**64
    if bucket < 0.70:
        return "train"
    if bucket < 0.85:
        return "val"
    return "test"

other_fauna["partition"] = other_fauna.apply(assign_candidate_partition, axis=1)
candidate_columns = [
    "filename", "primary_label", "class_name", "secondary_labels", "secondary_codes",
    "target_overlap", "candidate_status", "collection", "rating", "duration_s",
    "author", "partition", "url", "license",
]
candidate_manifest = other_fauna[candidate_columns].copy()
candidate_manifest.to_csv(EXPERIMENTO3_TABLES / "other_fauna_candidates.csv", index=False)

soundscapes_meta = pd.read_csv(PROCESSED_DIR / "train_soundscapes_meta.csv")
soundscapes_meta["candidate_windows_5s"] = (soundscapes_meta["duration_s"] // 5).astype(int)
soundscapes_meta["label_status"] = "unlabeled_not_supervised"
soundscapes_meta["domain_split"] = soundscapes_meta["site"].map(
    lambda site: (
        "train"
        if int.from_bytes(sha256(str(site).encode("utf-8")).digest()[:8], "big") / 2**64 < 0.70
        else (
            "val"
            if int.from_bytes(sha256(str(site).encode("utf-8")).digest()[:8], "big") / 2**64 < 0.85
            else "test"
        )
    )
)
soundscape_manifest = soundscapes_meta[
    ["filename", "site", "datetime", "duration_s", "sample_rate", "channels",
     "candidate_windows_5s", "label_status", "domain_split"]
].copy()
soundscape_manifest.to_csv(EXPERIMENTO3_TABLES / "soundscape_unlabeled_manifest.csv", index=False)

candidate_summary = (
    candidate_manifest.groupby(["class_name", "candidate_status", "partition"])
    .agg(recordings=("filename", "size"), authors=("author", "nunique"), hours=("duration_s", lambda x: x.sum() / 3600))
    .reset_index()
)
soundscape_summary = soundscape_manifest.groupby(["domain_split", "label_status"]).agg(
    files=("filename", "size"),
    sites=("site", "nunique"),
    candidate_windows=("candidate_windows_5s", "sum"),
)

print(f"Candidatos no aviares: {len(candidate_manifest):,}; contaminación conocida: {int(candidate_manifest.target_overlap.map(bool).sum())}")
print(f"Soundscapes: {len(soundscape_manifest):,}; ventanas potenciales sin etiqueta: {int(soundscape_manifest.candidate_windows_5s.sum()):,}")
print("Los candidatos débiles y soundscapes no se añaden al entrenamiento supervisado del baseline.")
display(candidate_summary)
display(soundscape_summary)


Candidatos no aviares: 904; contaminación conocida: 1
Soundscapes: 9,726; ventanas potenciales sin etiqueta: 116,712
Los candidatos débiles y soundscapes no se añaden al entrenamiento supervisado del baseline.


,class_name,candidate_status,partition,recordings,authors,hours
0,Amphibia,known_target_overlap_exclude_pure_negative,train,1,1,0.003062
1,Amphibia,weak_other_fauna_candidate_unverified,test,39,24,0.195568
2,Amphibia,weak_other_fauna_candidate_unverified,train,507,132,4.523907
3,Amphibia,weak_other_fauna_candidate_unverified,val,29,22,0.192297
4,Insecta,weak_other_fauna_candidate_unverified,test,2,1,0.078489
5,Insecta,weak_other_fauna_candidate_unverified,train,147,16,4.694000
6,Insecta,weak_other_fauna_candidate_unverified,val,5,2,0.125231
7,Mammalia,weak_other_fauna_candidate_unverified,test,25,13,0.273922
8,Mammalia,weak_other_fauna_candidate_unverified,train,132,65,1.289936
9,Mammalia,weak_other_fauna_candidate_unverified,val,17,15,0.089181


,,files,sites,candidate_windows
domain_split,label_status,,,
test,unlabeled_not_supervised,1900,11,22800
train,unlabeled_not_supervised,6610,38,79320
val,unlabeled_not_supervised,1216,7,14592


In [23]:
import pandas as pd2
sub_sp = pd.read_csv(PROCESSED_DIR / "subset_recordings.csv")
selected_counts = sub_sp.groupby("primary_label").size()

species_counts = other_fauna.groupby(["class_name", "primary_label"]).size().rename("recordings")

pooled_other_count = len(other_fauna)
def has_secondary_labels(value):
    if pd2.isna(value):
        return False
    return any(label for label in ast.literal_eval(value))

actual_secondary_count = int(other_fauna.secondary_labels.map(has_secondary_labels).sum())

comparison = pd2.DataFrame(
    {
        "measure": [
            "Selected Hito 1 bird recordings",
            "Number of selected bird classes",
            "Smallest selected bird class",
            "Largest selected bird class",
            "Other-fauna recordings after filters",
            "Other-fauna species",
            "Other-fauna species with <20 recordings",
            "Other-fauna recordings with secondary label(s)",
        ],
        "value": [
            len(sub_sp),
            selected_counts.size,
            selected_counts.min(),
            selected_counts.max(),
            pooled_other_count,
            other_fauna.primary_label.nunique(),
            int(species_counts.lt(20).sum()),
            int(actual_secondary_count),
        ],
    }
)
print(f"Desbalance 16 clases Hito 1: {selected_counts.max() / selected_counts.min():.2f}:1")
comparison

Desbalance 16 clases Hito 1: 6.77:1


,measure,value
0,Selected Hito 1 bird recordings,6759
1,Number of selected bird classes,16
2,Smallest selected bird class,145
3,Largest selected bird class,982
4,Other-fauna recordings after filters,904
5,Other-fauna species,60
6,Other-fauna species with <20 recordings,46
7,Other-fauna recordings with secondary label(s),5



### Interpretación y decisión de inclusión

- Conservar los `20 748` clips procesados del Hito 2 como baseline fijo.
- El pool `other_fauna_candidates.csv` permite experimentar con aprendizaje débil, pero sus filas no se consideran negativos verificados. Excluir `known_target_overlap_exclude_pure_negative`; mantener las demás con estado explícito de incertidumbre y menor peso en una prueba exploratoria. Reportar resultados en aves test sin incluir el pool débil en test.
- Mantener los soundscapes en un flujo aparte de datos no etiquetados. Un detector acústico general (p. ej., YAMNet) puede sugerir etiquetas ambientales como lluvia, agua o voz, pero esas etiquetas son débiles y no confirman ausencia de aves. Aplicar aprendizaje no supervisado/consistencia o pseudoetiquetado solo en train; reservar sitios para validación/test y no contabilizarlos como negativos de referencia.
- Para clasificación operacional, aplicar umbral de rechazo calibrado en la validación de las 16 aves. Una ventana debajo del umbral se reporta `reject/uncertain`, no necesariamente `otra_fauna`.
- Solo si un resultado débil mejora el rechazo de forma estable sin degradar el recall de las aves, justificar una clase agregada. Sin truth data independiente, la conclusión se limita a viabilidad técnica; no permite afirmar reducción real de falsas alarmas.

Los archivos de esta copia se escriben bajo `experiments/results/experimento_03/`; no se sobrescriben resultados ni datos procesados de Hito 2.
